# 🧠 TensorStack: Chapter 2 Masterclass
## Building Tokenizers & Vector Embeddings from Scratch in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![YouTube Video](https://img.shields.io/badge/YouTube-Watch%20Chapter%202-red)](https://youtu.be/050r1czRoxQ)

Welcome to the official interactive notebook for **Chapter 2 of the TensorStack LLM Masterclass**.
In this notebook, we implement the complete text processing and embedding pipeline in raw PyTorch without external high-level model wrappers.

### 1. Install & Import Dependencies

In [ ]:
!pip install tiktoken torch numpy

import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

print(f"✔ PyTorch Version: {torch.__version__}")

### 2. Building a Custom Regex Tokenizer from First Principles

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [token.strip() for token in tokens if token.strip()]
        token_ids = [
            self.str_to_int.get(token, self.str_to_int["<|unk|>"])
            for token in tokens
        ]
        return token_ids

    def decode(self, token_ids):
        tokens = [self.int_to_str[i] for i in token_ids]
        text = " ".join(tokens)
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        text = re.sub(r'\s+--\s+', '--', text)
        return text

# Build sample vocabulary
sample_text = "Hello, world! Building LLMs from scratch in PyTorch is amazing."
tokens = re.split(r'([,.:;?_!"()\']|--|\s)', sample_text)
tokens = [t.strip() for t in tokens if t.strip()]
vocab = {token: idx for idx, token in enumerate(sorted(list(set(tokens))) + ["<|endoftext|>", "<|unk|>"])}

tokenizer = SimpleTokenizerV2(vocab)
ids = tokenizer.encode("Hello, world! Building unknown_words.")
print("Encoded IDs:", ids)
print("Decoded Text:", tokenizer.decode(ids))

### 3. Production Tiktoken BPE Tokenizer

In [ ]:
enc = tiktoken.get_encoding("gpt2")
text = "Hello, world! Multi-Head Attention in PyTorch."
ids = enc.encode(text)
print(f"Total Tokens: {len(ids)}")
print(f"Token IDs:    {ids}")
print(f"Subword breakdown: {[enc.decode([i]) for i in ids]}")

### 4. PyTorch Sliding Window Dataset & DataLoader

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length=4, stride=1):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i : i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1 : i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

corpus = "In the beginning God created the heavens and the earth."
dataset = GPTDatasetV1(corpus, enc, max_length=4, stride=1)
loader = DataLoader(dataset, batch_size=2, shuffle=False)

inputs, targets = next(iter(loader))
print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)
print("Input batch 0:", inputs[0].tolist())
print("Target batch 0 (shifted by +1):", targets[0].tolist())

### 5. Token & Positional Embedding Layers

In [ ]:
class MasterEmbeddingPipeline(nn.Module):
    def __init__(self, vocab_size=50257, context_length=1024, d_model=768, drop_rate=0.1):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, d_model)
        self.position_embeddings = nn.Embedding(context_length, d_model)
        self.dropout = nn.Dropout(p=drop_rate)

    def forward(self, in_idx):
        batch_size, num_tokens = in_idx.shape
        tok_embeds = self.token_embeddings(in_idx)
        pos_ids = torch.arange(num_tokens, device=in_idx.device)
        pos_embeds = self.position_embeddings(pos_ids)
        # Superposition addition
        output = self.dropout(tok_embeds + pos_embeds)
        return output

pipeline = MasterEmbeddingPipeline(vocab_size=50257, context_length=1024, d_model=768)
output_tensor = pipeline(inputs)
print("🌟 Final 3D Input Tensor Shape:", output_tensor.shape)  # [Batch, Seq_Len, d_model]
print("✔ Ready for Chapter 3 Multi-Head Self-Attention!")